In [1]:
from pathlib import Path
import random
import shutil
import re

In [2]:
# Inputs
SOURCES = ["unhcr", "prwp", "refugee"]
TYPES = ["figure", "table"]
N_SAMPLES = 35  # per source x type
SNAPSHOTS_OUTPUT_PATH = "data/snapshots/"
METADATA_OUTPUT_PATH = "data/metadata/"
EXCLUDE_PATH = "../schema_discovery/data/snapshots/"

# Datasets accessible on HF:
# Snapshots - https://huggingface.co/datasets/ai4data/data-snapshot/tree/main/snapshots
# Metadata - https://huggingface.co/datasets/ai4data/data-snapshot/tree/main/metadata
SNAPSHOTS_DIR = "/home/ajd/hf_datasets/data-snapshot/snapshots/"
METADATA_DIR = "/home/ajd/hf_datasets/data-snapshot/metadata/"

In [3]:
excl_list = [x.name for x in Path(EXCLUDE_PATH).rglob("*.png")]
pat = (
    "^(?P<source_document_id>.+)_(?P<artifact_type>figure|table)_(?P<index>\d{3})\.png$"
)

# Sample snapshots
for s in SOURCES:
    all_snaps = list(Path(SNAPSHOTS_DIR + s).glob("*.png"))

    # Remove if part of the initial set
    all_snaps = [x for x in all_snaps if x.name not in excl_list]
    
    for t in TYPES:
        # Create output dir if not yet exsiting
        out_dir = Path(SNAPSHOTS_OUTPUT_PATH) / s / t
        out_dir.mkdir(exist_ok=True, parents=True)

        # Filter by type
        snaps_type = [
            x for x in all_snaps if re.search(pat, x.name).group("artifact_type") == t
        ]

        # Sample
        samples = random.sample(snaps_type, k=min(N_SAMPLES, len(snaps_type)))

        # Copy files
        for x in samples:
            shutil.copy(x, out_dir)

In [4]:
# Copy metadata
for s in SOURCES:
    # Create output dir if not yet exsiting
    out_dir = Path(METADATA_OUTPUT_PATH) / s
    out_dir.mkdir(exist_ok=True, parents=True)

    samples = list(Path(SNAPSHOTS_OUTPUT_PATH + s).glob("*/*.png"))

    # Create list of source document ids to copy
    to_copy = set()
    for x in samples:
        match = re.search(pat, x.name)
        to_copy.add(match.group("source_document_id"))

    # Copy files
    for x in to_copy:
        try:
            f = Path(METADATA_DIR) / s / (x + "_metadata.json")
            shutil.copy(f, out_dir)
        except FileNotFoundError:
            f = Path(METADATA_DIR) / s / (x + ".json")
            shutil.copy(f, out_dir)